# Heart Disease — Inference Demo

Loads the persisted `models/model.pkl` and runs predictions for single + batch patient inputs.

In [1]:
import sys
from pathlib import Path
import joblib
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

## 1. Load the persisted pipeline

In [2]:
model = joblib.load(ROOT / 'models' / 'model.pkl')
print('Pipeline steps:', [s for s, _ in model.steps])
print('Final estimator:', type(model.named_steps['model']).__name__)

Pipeline steps: ['preprocessor', 'model']
Final estimator: LogisticRegression


## 2. Single-patient prediction

In [3]:
sample = {
    'age': 54, 'sex': 1, 'cp': 0, 'trestbps': 130, 'chol': 250,
    'fbs': 0, 'restecg': 1, 'thalach': 150, 'exang': 0,
    'oldpeak': 1.0, 'slope': 1, 'ca': 0, 'thal': 2,
}
df_one = pd.DataFrame([sample])
proba = model.predict_proba(df_one)[0, 1]
label = int(model.predict(df_one)[0])
print(f'P(disease) = {proba:.3f} | predicted label = {label}')

P(disease) = 0.097 | predicted label = 0


## 3. Batch prediction

In [4]:
batch = pd.DataFrame([
    sample,
    {**sample, 'age': 35, 'chol': 180, 'thalach': 190, 'oldpeak': 0.0},
    {**sample, 'age': 70, 'chol': 320, 'oldpeak': 4.5, 'ca': 3},
], index=['baseline', 'low_risk', 'high_risk'])
batch['p_disease'] = model.predict_proba(batch)[:, 1].round(3)
batch['pred']      = model.predict(batch)
batch[['age', 'chol', 'thalach', 'oldpeak', 'ca', 'p_disease', 'pred']]

,age,chol,thalach,oldpeak,ca,p_disease,pred
baseline,54,250,150,1.0,0,0.097,0
low_risk,35,180,190,0.0,0,0.039,0
high_risk,70,320,150,4.5,3,0.657,1
